# Crystal Structure Analysis: Primitive vs. Conventional

This notebook demonstrates the differences between **primitive** and **conventional** crystal structures using the `pymatgen` library. It inspects materials from the project datasets to visualize how `SpacegroupAnalyzer` transformations affect unit cell definitions.

## 1. Concepts and Definitions

### Primitive Structure
**Method**: `get_primitive_standard_structure()`

*   **Definition**: According to the [official documentation](https://pymatgen.org/pymatgen.symmetry.html#pymatgen.symmetry.analyzer.SpacegroupAnalyzer.get_primitive_standard_structure): "Get a structure with a primitive cell according to certain standards. The standards are defined in Setyawan, W., & Curtarolo, S. (2010). High-throughput electronic band structure calculations: Challenges and tools. Computational Materials Science, 49(2), 299-312, [10.1016/j.commatsci.2010.05.010](https://doi.org/10.1016/j.commatsci.2010.05.010)."
*   **Characteristics**:
    *   **Volume**: Minimal.
    *   **Axes**: Often non-orthogonal (angles $\neq$ 90°), even for highly symmetric materials like cubic metals.
    *   **Geometry**: Does not always intuitively reflect the full symmetry (e.g., a cubic material might be represented as a rhombohedron).
*   **Use Case**: Generally preferred for **electronic structure calculations (DFT)** because fewer atoms mean significantly lower computational cost. *Note: In this project, FDMNES simulations use the **conventional structure** instead (see below).*

### Conventional Structure
**Method**: `get_conventional_standard_structure()`

*   **Definition**: According to the [official documentation](https://pymatgen.org/pymatgen.symmetry.html#pymatgen.symmetry.analyzer.SpacegroupAnalyzer.get_conventional_standard_structure): "The structure in a conventional standardized cell."
*   **Characteristics**:
    *   **Symmetrization**: **Yes**. Internally, this method calls `get_refined_structure()` first, ensuring that atoms are "snapped" to their ideal symmetry positions.
    *   **Standardization**: Applies **additional** standardization beyond `get_refined_structure()` by enforcing specific conventions for the lattice vectors (e.g., ordering axes such that $a < b < c$, or specific orientation rules defined by Setyawan & Curtarolo).
    *   **Volume**: Often larger (an integer multiple of the primitive cell).
    *   **Axes**: Aligned with the Cartesian axes where possible. For cubic, tetragonal, and orthorhombic systems, the angles are all 90°.
    *   **Geometry**: Intuitively displays the high symmetry (e.g., a cube looks like a cube).
*   **Use Case**: Preferred for **visualization**, **CIF file generation**, and standard interactions because humans find orthogonal axes easier to interpret.

### Refined Structure (`get_refined_structure`)

*   **Definition**: According to the [official documentation](https://pymatgen.org/pymatgen.symmetry.html#pymatgen.symmetry.analyzer.SpacegroupAnalyzer.get_refined_structure): "Get the refined structure based on detected symmetry. The refined structure is a conventional cell setting with atoms moved to the expected symmetry positions."
*   **Role**: 
    1.  **Standardization**: Converts the input structure (which might be primitive or non-standard) into the standard, recognizable conventional representation.
    2.  **Symmetrization**: Removes small distortions by enforcing exact symmetry on atomic coordinates.

### Application in FDMNES Workflow

In the project script `scripts/datasets/fdmnes/make_fdmnes_input_files.py`, structures are converted to CIF using:
```python
structure.to(cif_filename, symprec=0.01, angle_tolerance=5)
```
When `symprec` is provided, `pymatgen` defaults to writing the structure in its **conventional standard setting** (specifically calling `get_refined_structure()` internally). Therefore, **FDMNES simulations in this project are performed using the conventional structure**.

In [1]:
import pickle
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from xasml import resource_path

## 2. Load Material Data
We load the Iron (Fe) dataset to inspect specific examples.

In [2]:
path = resource_path("xasml:datasets/materials_project/pkls/Fe.pkl")
with open(path, "rb") as f:
    materials = pickle.load(f)

print(f"Loaded {len(materials)} materials.")

Loaded 12915 materials.


## 3. Structural Comparison (Iron Example)

We select **Iron (mp-13)** as a clear example. It is a Body-Centered Cubic (BCC) metal, so the difference between Primitive (1 atom) and Conventional (2 atoms) is stark.

In [3]:
target_id = "mp-13" 
target_material = next((m for m in materials if m["material_id"] == target_id), None)

if target_material:
    structure = target_material["structure"]
    print(f"--- Original Structure ({target_id}) ---")
    print(f"Lattice Angles: {structure.lattice.angles}\n")

    spa = SpacegroupAnalyzer(structure, symprec=0.01, angle_tolerance=5)

    refined = spa.get_refined_structure()
    print("--- Refined Structure (Used for FDMNES) ---")
    print(refined)
    print(f"Lattice: {refined.lattice.abc}")
    print(f"Angles : {refined.lattice.angles}\n")

    conventional = spa.get_conventional_standard_structure()
    print("--- Conventional Standard Structure ---")
    print(conventional)
    print(f"Lattice: {conventional.lattice.abc}")
    print(f"Angles : {conventional.lattice.angles}\n")

    primitive = spa.get_primitive_standard_structure()
    print("--- Primitive Standard Structure ---")
    print(primitive)
    print(f"Lattice: {primitive.lattice.abc}")
    print(f"Angles : {primitive.lattice.angles}\n")
else:
    print(f"Material {target_id} not found in the dataset.")

--- Original Structure (mp-13) ---
Lattice Angles: (90.00003845025343, 90.00031950636807, 109.469836683298)

--- Refined Structure (Used for FDMNES) ---
Full Formula (Fe2)
Reduced Formula: Fe
abc   :   2.863035   2.863035   2.863035
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (2)
  #  SP      a    b    c
---  ----  ---  ---  ---
  0  Fe    0    0    0
  1  Fe    0.5  0.5  0.5
Lattice: (2.863035498949916, 2.863035498949916, 2.863035498949916)
Angles : (90.0, 90.0, 90.0)

--- Conventional Standard Structure ---
Full Formula (Fe2)
Reduced Formula: Fe
abc   :   2.863035   2.863035   2.863035
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (2)
  #  SP      a    b    c
---  ----  ---  ---  ---
  0  Fe    0    0    0
  1  Fe    0.5  0.5  0.5
Lattice: (2.863035498949916, 2.863035498949916, 2.863035498949916)
Angles : (90.0, 90.0, 90.0)

--- Primitive Standard Structure ---
Full Formula (Fe1)
Reduced Formula: Fe

## 4. More Complex Example (mp-449)

Now we look at `mp-449` (Fe5Si3). Sometimes the primitive and conventional cells have the same number of atoms but different orientations.

In [4]:
target_id = "mp-449"
target_material = next((m for m in materials if m["material_id"] == target_id), None)

if target_material:
    structure = target_material["structure"]
    spa = SpacegroupAnalyzer(structure, symprec=0.01, angle_tolerance=5)

    refined = spa.get_refined_structure()
    primitive = spa.get_primitive_standard_structure()
    
    print(f"--- {target_id} ---")
    print(f"Refined (Conventional) Volume: {refined.volume:.3f} A^3, Sites: {refined.num_sites}")
    print(f"Primitive Volume             : {primitive.volume:.3f} A^3, Sites: {primitive.num_sites}")
    print(f"Checking Equivalence: {refined.volume / primitive.volume :.2f}x multiple")
else:
    print(f"Material {target_id} not found in the dataset.")

--- mp-449 ---
Refined (Conventional) Volume: 182.097 A^3, Sites: 16
Primitive Volume             : 182.097 A^3, Sites: 16
Checking Equivalence: 1.00x multiple
